In [1]:
from pathlib import Path
TRITON_API_KEY = Path('../api-key.txt').expanduser().read_text(encoding='utf-8').splitlines()[0].strip()
BASE_URL = 'https://tritonai-api.ucsd.edu'

FileNotFoundError: [Errno 2] No such file or directory: '../api-key.txt'

In [2]:
from typing import Dict, Any
from typing import List as listtype
import re

In [3]:
# document loaders
from langchain_community.document_loaders import DirectoryLoader, TextLoader, UnstructuredRSTLoader
from langchain_core.documents import Document

# text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# retriever
from langchain_core.retrievers import BaseRetriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import TFIDFRetriever, AmazonKnowledgeBasesRetriever

# cross encoder for reranking
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker


In [4]:
# rapidfire imports
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
from rapidfireai import Experiment

In [5]:
import rapidfireai.automl
print(dir(rapidfireai.automl))

['AutoMLAlgorithm', 'List', 'ModelConfig', 'RFDPOConfig', 'RFGRPOConfig', 'RFGridSearch', 'RFLangChainRagSpec', 'RFLoraConfig', 'RFModelConfig', 'RFOpenAIAPIModelConfig', 'RFPromptManager', 'RFRandomSearch', 'RFSFTConfig', 'RFvLLMModelConfig', 'Range', '_EVALS_CONFIGS_AVAILABLE', '_EVALS_HELPERS_AVAILABLE', '_FIT_CONFIGS_AVAILABLE', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'automl_utils', 'base', 'datatypes', 'get_flattened_config_leaf', 'get_runs', 'grid_search', 'model_config', 'random_search']


In [6]:
import rapidfireai.automl as automl

print("RFOpenAIAPIModelConfig:", automl.RFOpenAIAPIModelConfig)
print("RFLangChainRagSpec:", automl.RFLangChainRagSpec)
print("RFGridSearch:", automl.RFGridSearch)

RFOpenAIAPIModelConfig: None
RFLangChainRagSpec: <class 'rapidfireai.automl.model_config.RFLangChainRagSpec'>
RFGridSearch: <class 'rapidfireai.automl.grid_search.RFGridSearch'>


In [ ]:
type(RFOpenAIAPIModelConfig)

NoneType

[https://oss-docs.rapidfire.ai/en/latest/ragspecs.html](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

In [ ]:
class UnstructuredRSTLoaderWithMetadata(UnstructuredRSTLoader):
    def __init__(self, file_path: str, metadata_func=None, **kwargs):
        super().__init__(file_path=file_path, **kwargs)
        self.metadata_func = metadata_func
    
    def load(self) -> list[Document]:
        # load the documents as UnstructuredRSTLoader
        docs = super.load()
        # if a metadata function is specified
        if self.metadata_func is not None:
            for d in docs:
                d.metadata = self.metadata_func(d, d.metadata)
        
        return docs
    
    def lazy_load(self):
        # for each document loaded using UnstructuredRSTLoader
        for doc in super().lazy_load():
            # if a metadata function is specified
            if self.metadata_func is not None:
                # apply the metadata function to the document
                doc.metadata = self.metadata_func(doc, dict(doc.metadata))
            yield doc

In [ ]:
def metadata_func(document, metadata):
    # extract the page title from the content and store it in metadata
    title = document.page_content.split('\n')[0].split('===')[0]
    metadata['title'] = title
    return metadata

In [ ]:
loader = DirectoryLoader(
    path='../data/sourcedocs/',
    glob='**/*.rst',
    loader_cls=UnstructuredRSTLoaderWithMetadata,
    loader_kwargs={
        'metadata_func': metadata_func,
        'mode': 'single',
    },
    show_progress=True,
    use_multithreading=True
)

In [ ]:
docs = loader.load()

 35%|███▌      | 11/31 [00:02<00:03,  6.61it/s][WARNING] Could not convert TeX math \text{FPC} = \sqrt{(N-n)/(N-1)}, rendering as TeX
[WARNING] Could not convert TeX math \text{SE}_{\hat{p}} = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}} \times \text{FPC}, rendering as TeX
[WARNING] Could not convert TeX math \text{center} = \frac{\hat{p} + z^2/(2n_{\text{eff}})}{1 + z^2/n_{\text{eff}}}, rendering as TeX
[WARNING] Could not convert TeX math \text{margin} = \frac{z\sqrt{\hat{p}(1-\hat{p})/n_{\text{eff}} + z^2/(4n_{\text{eff}}^2)}}{1 + z^2/n_{\text{eff}}}, rendering as TeX
[WARNING] Could not convert TeX math \varepsilon = (b-a)\sqrt{\frac{\ln(2/\alpha)}{2n}} \times \text{FPC}, rendering as TeX
[WARNING] Could not convert TeX math \varepsilon_{\bar{X}} = R\sqrt{\ln(2/\alpha)/(2n)}, rendering as TeX

100%|██████████| 31/31 [00:04<00:00,  7.18it/s]


In [ ]:
docs[0]

Document(metadata={'source': '../data/sourcedocs/index.rst', 'title': 'Redirecting to overview...'}, page_content='Redirecting to overview...')

Models:
- `gpt2` / `r50k_base`: older ~50k-vocab GPT-2 style tokenizers. `r50k_base` is the standard GPT-2 BPE; `gpt2` is the GPT-2 encoding entry in `tiktoken`. Both use the same regex pattern and the same end-of-text token, with `gpt2` built from the original GPT-2 encoder/vocab files
- `p50k_base`: also ~50k vocab, but different merge ranks from `r50k_base`; intended for Codex-style/code-heavy tokenization
- `p50k_edit`: same base merges as `p50k_base`, but adds extra FIM/edit special tokens like `<|fim_prefix|>`, `<|fim_middle|>`, and `<|fim_suffix|>`
- `cl100k_base`: newer ~100k-vocab tokenizer with a different regex and a larger special-token set; commonly associated with GPT-4 / GPT-3.5-era model families
- `o200k_base`: newer ~200k-vocab tokenizer with a different regex again; used by newer model families such as `gpt-4o`
- `o200k_harmony`: derived from `o200k_base` but with many additional special tokens layered on top

In [ ]:
# text splitter 
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
     encoding_name='o200k_base',
     chunk_size=512, 
     chunk_overlap=64
)

From [RapidFire AI oss](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

`embedding_cfg (dict[str, Any], optional)`

The embedding class and its kwargs to convert a chunk/query into a vector, provided as a single dictionary. Must include a key `"class"` with the class itself as value, not an instance. Options for the class include `HuggingFaceEmbeddings` and `OpenAIEmbeddings`. 

The kwargs that follow must contain all parameters needed to initialize the embedding class; required parameters vary by embedding class. For example, `HuggingFaceEmbeddings` needs `model_name`, `model_kwargs` and `device`, while `OpenAIEmbeddings` needs `"model"` and `"api_key"`.

In [ ]:
batch_size = 32

embedding_cfg = List([
    # yet to use open ai embeddings
    {
        'class': HuggingFaceEmbeddings,
        'model_name': 'sentence-transformers/all-MiniLM-L6-v2',
        'model_kwargs': {'device': 'cpu'},
        'encode_kwargs': {'normalize_embeddings': True, 'batch_size': batch_size}
    }
])

From [RapidFire AI oss](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

`vector_store_cfg (dict[str, Any], optional)`

The vector store type and args to store and possibly index embedding vectors for retrieval, provided as a single dictionary.

- `"type"`: The type of vector store to use. Must be one of `"faiss"`, `"pgvector"`, or `"pinecone"`. Required.
- `"batch_size"`: Number of vectors per insert batch. Applies to all 3 types of stores. Optional; default is 128.

In [ ]:
vector_store_cfg={
    'type': 'faiss', 
    'batch_size': batch_size
}

In [ ]:
search_cfg=List([
    {'type': 'similarity', 'k': 10},
    {'type': 'mmr', 'k': 10}
])

In [ ]:
reranker_cfg={
    'class': CrossEncoderReranker,
    'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2',
    'model_kwargs': {'device': 'cpu'},
    'top_n': 5,
}

In [ ]:
def custom_template(doc):
    return f'{doc.metadata["title"]}: {doc.page_content}'

In [ ]:
rag = RFLangChainRagSpec(
    # to load the corpus
    document_loader=loader,
    # to split documents into chunks
    text_splitter=text_splitter,
    # to embed the chunks
    embedding_cfg=embedding_cfg,
    # vectore storage
    vector_store_cfg=vector_store_cfg,
    # search algorithm
    search_cfg=search_cfg,
    # rerank top results with a cross encoder
    reranker_cfg=reranker_cfg,
    # whether to use gpu-accelerated faiss
    enable_gpu_search=False,
    # to format each retrieved chunk for context injection into prompts
    document_template=custom_template,
)

In [ ]:
INSTRUCTIONS = """
You are a helpful assistant answering questions about RapidFire AI docuentation.

Your job:
- Read the retrieved documentation context
- Answer the user's question using only the context
- Do not make up information that is not supported by the retrieved text

Rules:
1. Use only the retrieved document context.
2. Keep the answer concise but complete.
3. Mention relevant file names when they are clearly useful.

Output format:
Reasoning: <brief reasoning grounded in the docs> 
#### ANSWER: <final answer>
"""

In [ ]:
import json
from datasets import Dataset

In [ ]:
def load_eval_question_set(path, encoding='utf-8', validation=False):
    # load the questions from the file
    with open(path, 'r', encoding=encoding) as f:
        eval_questions = json.load(f)
    
    # to store each example
    eval_rows = []

    for example in eval_questions:
        # extract the question and id
        row = {
            'question_id': example['question_id'],
            'question': example['question']
        }
        # validation sets include reference answer an source evidence
        if validation:
            row['reference_answer'] = example['reference_answer']
            row['source_evidence'] = example['source_evidence']
        
        eval_rows.append(row)

    
    # convert into a hugging face dataset
    eval_dataset = Dataset.from_list(eval_rows)
    return eval_dataset

In [ ]:
golden_qa_val = load_eval_question_set(path='../data/validation-set/validation-set-golden-qa-pairs.json', validation=True)
print(golden_qa_val)
print(golden_qa_val[1].keys())
print(golden_qa_val[1])

golden_qa = load_eval_question_set(path='../data/validation-set/validation-set-golden-qa-pairs.json')
print(golden_qa)
print(golden_qa[1].keys())
print(golden_qa[1])

Dataset({
    features: ['question_id', 'question', 'reference_answer', 'source_evidence'],
    num_rows: 45
})
dict_keys(['question_id', 'question', 'reference_answer', 'source_evidence'])
{'question_id': 2, 'question': "How does RapidFire AI's adaptive execution engine differ from traditional sequential execution for multi-config experiments?", 'reference_answer': "Traditional tools force you to run one config after another sequentially. In contrast, RapidFire AI's adaptive execution engine shards the dataset randomly into chunks and schedules all runs on one shard at a time, then cycles through all shards. This enables interruptible execution across GPUs/CPUs, providing a far more concurrent learning experience by automatically swapping adapters and base models across GPUs and DRAM via efficient shared memory-based caching mechanisms. The swapping overhead is minimal, less than 5% of runtime. This means before the third model config even starts in the sequential approach, RapidFire 

In [ ]:
def sample_preprocess_fn(batch: Dict[str, listtype], 
                         rag: RFLangChainRagSpec, 
                         prompt_manager: RFPromptManager) -> Dict[str, listtype]:
    '''
    convert a batch of dataset rows into model-ready prompts
    '''
    # retrieve context for each question in the batch
    all_context = rag.get_context(batch_queries=batch['question'], serialize=False)
    # extract the source metadata field from each retrieved document
    retrieved_source_files = [
        [doc.metadata['source'] for doc in docs] for docs in all_context
    ]
    # convert retrieved docs into plain text strings suitable for prompt injection
    serialized_context = rag.serialize_documents(all_context)

    # build a chat-style prompt per question
    prompts = [
        [
            # instruction as system prompt
            {'role': 'system', 'content': INSTRUCTIONS},
            # user message containing the question
            {
                'role': 'user', 'content': f'Question: {question} \nDocumentation context: {context} \nYour response:'
            }
        ]
        for question, context in zip(batch['question'], serialized_context)
    ]

    return {
        'prompts': prompts,
        'retrieved_source_files': retrieved_source_files,
        **batch
    }

In [ ]:
def extract_solution(answer: str) -> str:
    '''
    extract the answer
    '''
    # search for the answer marker
    match = re.search(
        r"####\s*ANSWER:\s*(.*)",
        answer,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if match:
        return match.group(1).strip()
    # if the marker is missing fall back to returning the whole stripped text
    return answer.strip()

In [ ]:
def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    '''
    turn raw generated text into a field
    '''
    batch['answer'] = [extract_solution(answer) for answer in batch['generated_text']]
    return batch

In [ ]:
openai_config = RFOpenAIAPIModelConfig(
    client_config={'api_key': TRITON_API_KEY, 'base_url': BASE_URL, 'max_retries': 2},
    model_config={
        'model': 'api-mistral-small-3.2-2506',
        'max_completion_tokens': 2048
    },
    # request per minute needs to be set based on your account tier and the specific model used
    rpm_limit=120,
    # token per minute needs to be set based on your account tier and the specific model used
    tpm_limit=1_000_000,
    # rag model
    rag=rag,
    prompt_manager=None
)

TypeError: 'NoneType' object is not callable

In [ ]:
# skip metrics for now
config_set = List([{
    'openai_config': openai_config,
    'batch_size': batch_size,
    'preprocess_fn': sample_preprocess_fn,
    'postprocess_fn': sample_postprocess_fn,
    # add metrics later
    # 'compute_metrics_fn': ...,
    # 'accumulate_metrics_fn': ...,
    # 'online_strategy_kwargs': {
    #     'strategy_name': 'normal',
    #     'confidence_level': 0.95,
    #     'use_fpc': True,
    # }
}])

config_group = RFGridSearch(config_set)

NameError: name 'openai_config' is not defined

In [ ]:
# create experiment
experiment = Experiment(
    experiment_name='rag-generation-only',
    model='evals'
)

# run generation
results = experiment.run_evals(
    config_group=config_group,
    dataset=golden_qa,
    num_shards=4,
    seed=42
)

results

In [ ]:
# inspect results
print(type(results))
print(results)
experiment.end()